In [1]:
from google.colab import drive

drive.mount('/content/drive')

print("✅ GOOGLE DRIVE MOUNTED")

Mounted at /content/drive
✅ GOOGLE DRIVE MOUNTED


In [2]:
import os
import numpy as np
import pandas as pd
from scipy import sparse
import joblib

BASE = "/content/drive/MyDrive/NewsGuard"

# -----------------------------
# Day 2: Preprocessed datasets
# -----------------------------
train_df = pd.read_csv(
    f"{BASE}/data/processed/train_preprocessed.csv"
)
val_df = pd.read_csv(
    f"{BASE}/data/processed/validation_preprocessed.csv"
)
test_df = pd.read_csv(
    f"{BASE}/data/processed/test_preprocessed.csv"
)

# -----------------------------
# Day 3: TF-IDF features
# -----------------------------
X_train_tfidf = sparse.load_npz(
    f"{BASE}/features/tfidf/X_train_tfidf.npz"
)
X_val_tfidf = sparse.load_npz(
    f"{BASE}/features/tfidf/X_validation_tfidf.npz"
)
X_test_tfidf = sparse.load_npz(
    f"{BASE}/features/tfidf/X_test_tfidf.npz"
)

# -----------------------------
# Day 4: Auxiliary features
# -----------------------------
X_train_aux = np.load(
    f"{BASE}/features/auxiliary/X_train_auxiliary.npy"
)
X_val_aux = np.load(
    f"{BASE}/features/auxiliary/X_validation_auxiliary.npy"
)
X_test_aux = np.load(
    f"{BASE}/features/auxiliary/X_test_auxiliary.npy"
)

# -----------------------------
# Day 5: Combined features
# -----------------------------
X_train_combined = sparse.load_npz(
    f"{BASE}/features/combined/X_train_combined.npz"
)
X_val_combined = sparse.load_npz(
    f"{BASE}/features/combined/X_validation_combined.npz"
)
X_test_combined = sparse.load_npz(
    f"{BASE}/features/combined/X_test_combined.npz"
)

# -----------------------------
# Day 5: NB features
# -----------------------------
X_train_nb = sparse.hstack(
    [X_train_tfidf, sparse.csr_matrix(
        np.load(f"{BASE}/features/auxiliary/X_train_auxiliary.npy")
    )],
    format="csr"
)

# -----------------------------
# Labels
# -----------------------------
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

# -----------------------------
# Day 5 baseline results
# -----------------------------
baseline_results = pd.read_csv(
    f"{BASE}/results/day_05_baseline_results.csv"
)

# -----------------------------
# Verification
# -----------------------------
print("DAY 1–5 ARTIFACT RESTORE")
print("=" * 60)

print(f"Train CSV       : {train_df.shape}")
print(f"Validation CSV  : {val_df.shape}")
print(f"Test CSV        : {test_df.shape}")

print(f"\nTrain TF-IDF    : {X_train_tfidf.shape}")
print(f"Validation TF-IDF: {X_val_tfidf.shape}")
print(f"Test TF-IDF     : {X_test_tfidf.shape}")

print(f"\nTrain Auxiliary : {X_train_aux.shape}")
print(f"Validation Aux  : {X_val_aux.shape}")
print(f"Test Auxiliary  : {X_test_aux.shape}")

print(f"\nTrain Combined  : {X_train_combined.shape}")
print(f"Validation Comb.: {X_val_combined.shape}")
print(f"Test Combined   : {X_test_combined.shape}")

print("\nLabels:")
print(f"Train: {y_train.shape}")
print(f"Val  : {y_val.shape}")
print(f"Test : {y_test.shape}")

print("\nDay 5 Baseline Results:")
display(baseline_results)

# Assertions
assert train_df.shape[0] == 31282
assert val_df.shape[0] == 3910
assert test_df.shape[0] == 3911

assert X_train_tfidf.shape == (31282, 5000)
assert X_val_tfidf.shape == (3910, 5000)
assert X_test_tfidf.shape == (3911, 5000)

assert X_train_aux.shape == (31282, 115)
assert X_val_aux.shape == (3910, 115)
assert X_test_aux.shape == (3911, 115)

assert X_train_combined.shape == (31282, 5115)
assert X_val_combined.shape == (3910, 5115)
assert X_test_combined.shape == (3911, 5115)

assert len(baseline_results) == 2

print("\n" + "=" * 60)
print("✅ DAY 1–5 ARTIFACT RESTORE & VERIFICATION PASSED")
print("=" * 60)

DAY 1–5 ARTIFACT RESTORE
Train CSV       : (31282, 7)
Validation CSV  : (3910, 7)
Test CSV        : (3911, 7)

Train TF-IDF    : (31282, 5000)
Validation TF-IDF: (3910, 5000)
Test TF-IDF     : (3911, 5000)

Train Auxiliary : (31282, 115)
Validation Aux  : (3910, 115)
Test Auxiliary  : (3911, 115)

Train Combined  : (31282, 5115)
Validation Comb.: (3910, 5115)
Test Combined   : (3911, 5115)

Labels:
Train: (31282,)
Val  : (3910,)
Test : (3911,)

Day 5 Baseline Results:


,Model,CV_Accuracy,CV_Precision,CV_Recall,CV_F1,CV_ROC_AUC
0,Logistic Regression,0.993383,0.993520,0.994280,0.993899,0.999408
1,Naive Bayes,0.953296,0.952792,0.961491,0.957115,0.990242



✅ DAY 1–5 ARTIFACT RESTORE & VERIFICATION PASSED


In [3]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

import pandas as pd
import numpy as np
import joblib
import time

# Stratified 5-Fold CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Primary metric for model comparison
SCORING = "f1"

print("✅ DAY 6 MODEL SETUP READY")
print("CV       :", cv)
print("Scoring  :", SCORING)
print("Models   : SVM, Random Forest, Gradient Boosting, XGBoost")

✅ DAY 6 MODEL SETUP READY
CV       : StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Scoring  : f1
Models   : SVM, Random Forest, Gradient Boosting, XGBoost


In [4]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate

svm_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=5000
)

start_time = time.time()

svm_cv = cross_validate(
    svm_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=-1,
    return_train_score=False
)

svm_time = time.time() - start_time

svm_result = {
    "Model": "Linear SVM",
    "CV_Accuracy": svm_cv["test_accuracy"].mean(),
    "CV_Precision": svm_cv["test_precision"].mean(),
    "CV_Recall": svm_cv["test_recall"].mean(),
    "CV_F1": svm_cv["test_f1"].mean(),
    "CV_ROC_AUC": svm_cv["test_roc_auc"].mean(),
    "F1_Std": svm_cv["test_f1"].std(),
    "Time_Seconds": svm_time
}

print("LINEAR SVM — 5-FOLD CV")
print("=" * 60)

for key, value in svm_result.items():
    if key == "Model":
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value:.6f}")

print("\nFold F1 Scores:")
print(np.round(svm_cv["test_f1"], 6))

print(f"\n⏱️ Time: {svm_time:.2f} seconds")
print("✅ LINEAR SVM CV COMPLETED")

LINEAR SVM — 5-FOLD CV
Model: Linear SVM
CV_Accuracy: 0.996420
CV_Precision: 0.996054
CV_Recall: 0.997346
CV_F1: 0.996700
CV_ROC_AUC: 0.999659
F1_Std: 0.000935
Time_Seconds: 41.659245

Fold F1 Scores:
[0.997348 0.997199 0.995579 0.99558  0.997791]

⏱️ Time: 41.66 seconds
✅ LINEAR SVM CV COMPLETED


In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

start_time = time.time()

rf_cv = cross_validate(
    rf_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=1,
    return_train_score=False
)

rf_time = time.time() - start_time

rf_result = {
    "Model": "Random Forest",
    "CV_Accuracy": rf_cv["test_accuracy"].mean(),
    "CV_Precision": rf_cv["test_precision"].mean(),
    "CV_Recall": rf_cv["test_recall"].mean(),
    "CV_F1": rf_cv["test_f1"].mean(),
    "CV_ROC_AUC": rf_cv["test_roc_auc"].mean(),
    "F1_Std": rf_cv["test_f1"].std(),
    "Time_Seconds": rf_time
}

print("RANDOM FOREST — 5-FOLD CV")
print("=" * 60)

for key, value in rf_result.items():
    if key == "Model":
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value:.6f}")

print("\nFold F1 Scores:")
print(np.round(rf_cv["test_f1"], 6))

print(f"\n⏱️ Time: {rf_time:.2f} seconds")
print("✅ RANDOM FOREST CV COMPLETED")

RANDOM FOREST — 5-FOLD CV
Model: Random Forest
CV_Accuracy: 0.990793
CV_Precision: 0.988228
CV_Recall: 0.994870
CV_F1: 0.991537
CV_ROC_AUC: 0.999617
F1_Std: 0.001069
Time_Seconds: 232.939802

Fold F1 Scores:
[0.990445 0.992643 0.991192 0.990451 0.992952]

⏱️ Time: 232.94 seconds
✅ RANDOM FOREST CV COMPLETED


In [9]:
from sklearn.feature_selection import SelectKBest, chi2
from scipy import sparse

# Use TF-IDF only for feature selection
# Auxiliary features will be added after selection

k_features = 500

selector_gb = SelectKBest(
    score_func=chi2,
    k=k_features
)

X_train_tfidf_gb = selector_gb.fit_transform(
    X_train_tfidf,
    y_train
)

X_val_tfidf_gb = selector_gb.transform(
    X_val_tfidf
)

X_test_tfidf_gb = selector_gb.transform(
    X_test_tfidf
)

# Add auxiliary features
X_train_gb = sparse.hstack(
    [
        X_train_tfidf_gb,
        sparse.csr_matrix(X_train_aux)
    ],
    format="csr"
)

X_val_gb = sparse.hstack(
    [
        X_val_tfidf_gb,
        sparse.csr_matrix(X_val_aux)
    ],
    format="csr"
)

X_test_gb = sparse.hstack(
    [
        X_test_tfidf_gb,
        sparse.csr_matrix(X_test_aux)
    ],
    format="csr"
)

print("GRADIENT BOOSTING FEATURES")
print("=" * 60)
print("Train:", X_train_gb.shape)
print("Validation:", X_val_gb.shape)
print("Test:", X_test_gb.shape)

assert X_train_gb.shape == (31282, 615)
assert X_val_gb.shape == (3910, 615)
assert X_test_gb.shape == (3911, 615)

print("\n✅ REDUCED FEATURES READY")

GRADIENT BOOSTING FEATURES
Train: (31282, 615)
Validation: (3910, 615)
Test: (3911, 615)

✅ REDUCED FEATURES READY


In [10]:
from sklearn.model_selection import StratifiedKFold

gb_cv_split = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

gb_model = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

start_time = time.time()

gb_cv = cross_validate(
    gb_model,
    X_train_gb.toarray(),
    y_train,
    cv=gb_cv_split,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=-1,
    return_train_score=False
)

gb_time = time.time() - start_time

gb_result = {
    "Model": "Gradient Boosting",
    "CV_Accuracy": gb_cv["test_accuracy"].mean(),
    "CV_Precision": gb_cv["test_precision"].mean(),
    "CV_Recall": gb_cv["test_recall"].mean(),
    "CV_F1": gb_cv["test_f1"].mean(),
    "CV_ROC_AUC": gb_cv["test_roc_auc"].mean(),
    "F1_Std": gb_cv["test_f1"].std(),
    "Time_Seconds": gb_time
}

print("GRADIENT BOOSTING — 3-FOLD CV")
print("=" * 60)

for key, value in gb_result.items():
    if key == "Model":
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value:.6f}")

print("\nFold F1 Scores:")
print(np.round(gb_cv["test_f1"], 6))

print(f"\n⏱️ Time: {gb_time:.2f} seconds")
print("✅ GRADIENT BOOSTING CV COMPLETED")

GRADIENT BOOSTING — 3-FOLD CV
Model: Gradient Boosting
CV_Accuracy: 0.995844
CV_Precision: 0.994301
CV_Recall: 0.998054
CV_F1: 0.996174
CV_ROC_AUC: 0.999716
F1_Std: 0.000272
Time_Seconds: 260.367483

Fold F1 Scores:
[0.996556 0.995939 0.996027]

⏱️ Time: 260.37 seconds
✅ GRADIENT BOOSTING CV COMPLETED


In [11]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

xgb_cv = cross_validate(
    xgb_model,
    X_train_gb,
    y_train,
    cv=gb_cv_split,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=1,
    return_train_score=False
)

xgb_time = time.time() - start_time

xgb_result = {
    "Model": "XGBoost",
    "CV_Accuracy": xgb_cv["test_accuracy"].mean(),
    "CV_Precision": xgb_cv["test_precision"].mean(),
    "CV_Recall": xgb_cv["test_recall"].mean(),
    "CV_F1": xgb_cv["test_f1"].mean(),
    "CV_ROC_AUC": xgb_cv["test_roc_auc"].mean(),
    "F1_Std": xgb_cv["test_f1"].std(),
    "Time_Seconds": xgb_time
}

print("XGBOOST — 3-FOLD CV")
print("=" * 60)

for key, value in xgb_result.items():
    if key == "Model":
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value:.6f}")

print("\nFold F1 Scores:")
print(np.round(xgb_cv["test_f1"], 6))

print(f"\n⏱️ Time: {xgb_time:.2f} seconds")
print("✅ XGBOOST CV COMPLETED")

XGBOOST — 3-FOLD CV
Model: XGBoost
CV_Accuracy: 0.997507
CV_Precision: 0.997114
CV_Recall: 0.998290
CV_F1: 0.997701
CV_ROC_AUC: 0.999954
F1_Std: 0.000260
Time_Seconds: 34.000478

Fold F1 Scores:
[0.998055 0.997613 0.997436]

⏱️ Time: 34.00 seconds
✅ XGBOOST CV COMPLETED


In [12]:
from sklearn.model_selection import GridSearchCV

xgb_grid_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [80, 120],
    "max_depth": [3, 4],
    "learning_rate": [0.1]
}

xgb_grid = GridSearchCV(
    estimator=xgb_grid_model,
    param_grid=xgb_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=1,
    verbose=1,
    return_train_score=False
)

start_time = time.time()

xgb_grid.fit(
    X_train_gb,
    y_train
)

xgb_grid_time = time.time() - start_time

print("XGBOOST — GRIDSEARCHCV")
print("=" * 60)
print("Best Parameters:")
print(xgb_grid.best_params_)

print(f"\nBest CV F1: {xgb_grid.best_score_:.6f}")
print(f"⏱️ Time: {xgb_grid_time:.2f} seconds")

print("\n✅ XGBOOST GRIDSEARCH COMPLETED")

Fitting 3 folds for each of 4 candidates, totalling 12 fits
XGBOOST — GRIDSEARCHCV
Best Parameters:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 120}

Best CV F1: 0.997672
⏱️ Time: 134.60 seconds

✅ XGBOOST GRIDSEARCH COMPLETED


In [13]:
xgb_tuning_result = {
    "Model": "XGBoost",
    "Best_CV_F1": xgb_grid.best_score_,
    "Best_Params": str(xgb_grid.best_params_),
    "GridSearch_Time_Seconds": xgb_grid_time
}

xgb_tuning_df = pd.DataFrame([xgb_tuning_result])

display(xgb_tuning_df)

xgb_tuning_df.to_csv(
    f"{BASE}/results/day_06_xgboost_gridsearch.csv",
    index=False
)

joblib.dump(
    xgb_grid.best_estimator_,
    f"{BASE}/results/day_06_xgboost_best_tuned.joblib"
)

print("✅ XGBoost tuning results saved")
print("✅ Best tuned XGBoost model saved")

,Model,Best_CV_F1,Best_Params,GridSearch_Time_Seconds
0,XGBoost,0.997672,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",134.595112


✅ XGBoost tuning results saved
✅ Best tuned XGBoost model saved


In [14]:
day_06_results = pd.DataFrame([
    svm_result,
    rf_result,
    gb_result,
    {
        "Model": "XGBoost",
        "CV_Accuracy": xgb_cv["test_accuracy"].mean(),
        "CV_Precision": xgb_cv["test_precision"].mean(),
        "CV_Recall": xgb_cv["test_recall"].mean(),
        "CV_F1": xgb_cv["test_f1"].mean(),
        "CV_ROC_AUC": xgb_cv["test_roc_auc"].mean(),
        "F1_Std": xgb_cv["test_f1"].std(),
        "Time_Seconds": xgb_time
    }
])

# Add Day 5 baseline models
comparison_results = pd.concat(
    [baseline_results, day_06_results],
    ignore_index=True
)

comparison_results = comparison_results.sort_values(
    by="CV_F1",
    ascending=False
).reset_index(drop=True)

print("NEWSGUARD — MODEL COMPARISON")
print("=" * 80)

display(comparison_results)

print("\n🏆 BEST MODEL:")
print(comparison_results.iloc[0]["Model"])
print(f"CV F1: {comparison_results.iloc[0]['CV_F1']:.6f}")

comparison_results.to_csv(
    f"{BASE}/results/day_06_model_comparison.csv",
    index=False
)

joblib.dump(
    comparison_results,
    f"{BASE}/results/day_06_model_comparison.joblib"
)

print("\n✅ DAY 5 + DAY 6 MODEL COMPARISON SAVED")

NEWSGUARD — MODEL COMPARISON


,Model,CV_Accuracy,CV_Precision,CV_Recall,CV_F1,CV_ROC_AUC,F1_Std,Time_Seconds
0,XGBoost,0.997507,0.997114,0.998290,0.997701,0.999954,0.000260,34.000478
1,Linear SVM,0.996420,0.996054,0.997346,0.996700,0.999659,0.000935,41.659245
2,Gradient Boosting,0.995844,0.994301,0.998054,0.996174,0.999716,0.000272,260.367483
3,Logistic Regression,0.993383,0.993520,0.994280,0.993899,0.999408,NaN,NaN
4,Random Forest,0.990793,0.988228,0.994870,0.991537,0.999617,0.001069,232.939802
5,Naive Bayes,0.953296,0.952792,0.961491,0.957115,0.990242,NaN,NaN



🏆 BEST MODEL:
XGBoost
CV F1: 0.997701

✅ DAY 5 + DAY 6 MODEL COMPARISON SAVED
